# **Notebook 3: Baseline Model Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] Internet access (to download the model)
- [ ] GPU runtime enabled (Runtime → Change runtime type → T4 GPU)

**Files this notebook will CREATE:**
- [ ] `outputs.json` — `test_query`, `ground_truth`, `baseline_output` _(Required by NB4, NB5, NB7)_

---

### Setup
Installs dependencies, defines helpers and (for Notebooks 3-7) loads the shared `config.json` written by Notebook 2. Run this cell first.

In [ ]:
# ---- Environment setup (Google Colab / local Jupyter) ----
import os, sys, glob, zipfile, json, random, warnings
warnings.filterwarnings("ignore")

USE_DRIVE = False   # set True on Colab to keep artifacts (chroma_db, adapters, csv) across runtime resets
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/GenAI_Capstone"
    os.makedirs(WORKDIR, exist_ok=True)
    os.chdir(WORKDIR)
print("Working directory:", os.getcwd())

%pip install -q transformers accelerate bitsandbytes pandas numpy

# ---- Load the single-source-of-truth configuration written by Notebook 2 ----
if not os.path.exists("config.json"):
    raise FileNotFoundError("config.json not found - run Notebook 2 (Data Preparation) first.")
cfg = json.load(open("config.json"))
MODEL_ID       = cfg["MODEL_ID"]
EMBED_MODEL    = cfg["EMBED_MODEL"]
SEED           = cfg["SEED"]
BASELINE_SYSTEM = cfg["BASELINE_SYSTEM"]
RAG_SYSTEM     = cfg["RAG_SYSTEM"]          # contains the {context} placeholder
ROUTER_SYSTEM  = cfg["ROUTER_SYSTEM"]
MAX_NEW_TOKENS = cfg["MAX_NEW_TOKENS"]
CHROMA_DIR     = cfg["CHROMA_DIR"]
COLLECTION     = cfg["COLLECTION"]
random.seed(SEED)
print("MODEL_ID:", MODEL_ID, "| EMBED_MODEL:", EMBED_MODEL, "| SEED:", SEED)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

HAS_CUDA = torch.cuda.is_available()
COMPUTE_DTYPE = torch.bfloat16 if (HAS_CUDA and torch.cuda.get_device_capability()[0] >= 8) else torch.float16
print("CUDA available:", HAS_CUDA, "| compute dtype:", COMPUTE_DTYPE)
if not HAS_CUDA:
    print("WARNING: no GPU found. Runtime -> Change runtime type -> T4 GPU. Falling back to (slow) CPU float32.")

def load_base_model(model_id=None, quantize=True):
    """Load tokenizer + base model. 4-bit NF4 quantisation on GPU (about 1 GB VRAM for the 1.5B model)."""
    model_id = model_id or MODEL_ID
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = "left"                       # required for batched decoder-only generation
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    if HAS_CUDA and quantize:
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=COMPUTE_DTYPE)
        model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map={"": 0})
    elif HAS_CUDA:
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=COMPUTE_DTYPE, device_map={"": 0})
    else:
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)
    model.eval()
    return tok, model

@torch.no_grad()
def generate_chat(model, tok, messages_list, max_new_tokens=None, batch_size=16):
    """Deterministic (greedy) batched generation. do_sample=False with temperature/top_p/top_k=None
    (never temperature=0.0 together with do_sample=False). Returns only the newly generated text."""
    max_new_tokens = max_new_tokens or MAX_NEW_TOKENS
    outs = []
    for i in range(0, len(messages_list), batch_size):
        batch = messages_list[i:i + batch_size]
        prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in batch]
        enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             temperature=None, top_p=None, top_k=None, pad_token_id=tok.pad_token_id)
        new_tokens = gen[:, enc["input_ids"].shape[1]:]
        outs += [t.strip() for t in tok.batch_decode(new_tokens, skip_special_tokens=True)]
    return outs

def baseline_messages(query):
    return [{"role": "system", "content": BASELINE_SYSTEM}, {"role": "user", "content": query}]

def rag_messages(query, context):
    return [{"role": "system", "content": RAG_SYSTEM.format(context=context)}, {"role": "user", "content": query}]

## **Stage 3: Solution V1 (Retrieval-Assisted Generation)**

### **Task 3.1: Establish Baseline Performance**

#### **3.1.1 Execute Baseline Inference [2 marks]**
**The Task:** Load the pre-trained base model in 4-bit quantization and generate a response to an ambiguous shipping-delay query without any context.

**Hints & Tips:**
* Use `do_sample=False` for deterministic output. Do NOT pair `temperature=0.0` with `do_sample=False` — it throws a deprecation warning. Use `temperature=None, top_p=None`.
* `BitsAndBytesConfig(load_in_4bit=True)` shrinks the 1.5B model to ~750MB VRAM.
* `max_new_tokens=120` gives room for a complete answer.

**Model Selection:**
* **Qwen/Qwen2.5-1.5B-Instruct** (recommended) — must match what you used in NB2.
* **TinyLlama-1.1B-Chat** — lighter, weaker structured output.
* **Llama-3-8B-Instruct** — best quality, may OOM on free T4 during fine-tuning.

**Learner Inference:** This establishes your zero-shot baseline. Every later improvement is measured against this exact output.

In [ ]:
import time
# 4-bit NF4 quantisation: the 1.5B model needs ~1 GB of VRAM instead of ~3 GB
tokenizer, base_model = load_base_model(MODEL_ID, quantize=True)
if HAS_CUDA:
    print(f"Loaded {MODEL_ID} in 4-bit | GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# An ambiguous / noisy shipping-delay query and the known SOP rule (ground truth)
test_query = ("I ordered a package over a week ago and it still hasn't arrived and the tracking hasn't moved either. "
              "This is getting ridiculous - how long is delivery supposed to take?")
ground_truth = "Domestic orders deliver within 3-7 business days."
print("test_query  :", test_query)
print("ground_truth:", ground_truth)

In [ ]:
t0 = time.time()
# deterministic decoding: do_sample=False with temperature=None / top_p=None (NOT temperature=0.0)
baseline_output = generate_chat(base_model, tokenizer, [baseline_messages(test_query)], max_new_tokens=MAX_NEW_TOKENS, batch_size=1)[0]
print(f"Generated in {time.time()-t0:.1f}s\n")
print("BASELINE OUTPUT (zero-shot, no context):\n" + "-" * 70 + "\n" + baseline_output + "\n" + "-" * 70)

#### **3.1.2 Evaluate Baseline Quality [2 marks]**
**The Task:** Assess the baseline output for factual inaccuracies against the ground-truth SOP rule.

**Hints & Tips:**
* Compare against the known rule: "Domestic orders deliver within 3-7 business days."
* Did the model invent a timeline? Mention a non-existent tracking system or department?
* Document every hallucination — it justifies Stages 3 and 4.

**Learner Inference:** This hallucination is exactly why you build Stage 3 (a database) and Stage 4 (a router).

In [ ]:
import re

# ---- Hallucination assessment of the baseline answer against the SOP rule (3-7 business days) --------
text = baseline_output
clean = re.sub(r"(?m)^\s*\d+[.)]\s+", "", text)                        # ignore list numbering
claims = re.findall(r"\d+\s*(?:-|to|–)\s*\d+\s*(?:business\s+|working\s+)?(?:days?|weeks?|hours?)|\b\d+\s*(?:business\s+|working\s+)?(?:days?|weeks?|hours?)", clean, flags=re.I)
states_correct = bool(re.search(r"\b3\s*(?:-|to|–)\s*7\b", clean))
invented = re.findall(r"\b(?:[A-Z][A-Za-z]+\s){1,3}(?:Department|Team|Portal|System|Hotline|Tracker|Desk)\b", clean)
contacts = re.findall(r"https?://\S+|www\.\S+|[\w.+-]+@[\w-]+\.[\w.]+|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b", clean)
overpromise = re.findall(r"\b(?:guarantee[ds]?|definitely|promise[ds]?|100%)\b", clean, flags=re.I)

print("Time-frame claims in the answer :", claims or "none")
print("States the correct 3-7 business-day rule:", states_correct)
print("Invented departments/systems     :", invented or "none")
print("Contact details / URLs invented  :", contacts or "none")
print("Over-promising language          :", overpromise or "none")

wrong_timeline = bool(claims) and not states_correct
hallucinated = wrong_timeline or bool(invented) or bool(contacts) or bool(overpromise)
if hallucinated:
    verdict = "HALLUCINATION DETECTED"
elif not states_correct:
    verdict = "NOT FACTUALLY CORRECT - generic answer without the SOP rule (no policy-specific facts)"
else:
    verdict = "Matches the SOP rule"
print("\nVerdict:", verdict)

# ---- output consistency (3 identical greedy runs) --------------------------------------------------
runs = [generate_chat(base_model, tokenizer, [baseline_messages(test_query)], batch_size=1)[0] for _ in range(3)]
print("Deterministic across 3 runs:", len(set(runs)) == 1)

assessment = {"states_correct_rule": states_correct, "timeline_claims": claims, "invented_entities": invented,
              "invented_contacts": contacts, "over_promises": overpromise, "verdict": verdict, "deterministic_3_runs": len(set(runs)) == 1}
print("\nManual review note: read the answer above and record every invented timeline / system / department.")
print("Learner inference: without the SOP the model cannot know the real 3-7 business-day rule - this is why Stage 3 (database) and Stage 4 (router) are built.")

---
## Save Artifacts for Downstream Notebooks

**IMPORTANT:** Saves the baseline output. Notebooks 4, 5, and 7 depend on this file.

In [ ]:
outputs = {"test_query": test_query, "ground_truth": ground_truth, "baseline_output": baseline_output,
           "baseline_assessment": assessment, "model_id": MODEL_ID}
json.dump(outputs, open("outputs.json", "w"), indent=2)
print("Saved outputs.json with keys:", list(json.load(open("outputs.json")).keys()))

---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 4.**

- [ ] Base model loaded in 4-bit without errors
- [ ] Baseline output generated for `test_query`
- [ ] Hallucination assessment documented
- [ ] **`outputs.json` saved** with `test_query`, `ground_truth`, `baseline_output` ← _CRITICAL for NB4, 5, 7_
- [ ] GPU runtime enabled

**If any item is unchecked, fix it before moving on.**